In [0]:
%sql
-- =============================================================================
-- Tabela.......: gold.mart_produto_credito
-- Camada.......: Gold
-- Tipo.........: Data Product (Materialized Data Mart)
--
-- Objetivo.....:
-- Consolidar indicadores da carteira por modalidade de crédito,
-- permitindo avaliar desempenho financeiro, exposição ao risco e
-- inadimplência de cada produto.
--
-- Consumidores.:
-- • Gerência Comercial
-- • Gerência de Produtos
-- • Gerência de Crédito
-- • Diretoria
--
-- Grão.........:
-- 1 registro por Produto de Crédito.
--
-- Fonte........:
-- gold.fact_carteira_credito
-- gold.dim_produto
--
-- KPIs.........:
-- • Quantidade de contratos
-- • Valor contratado
-- • Participação da carteira
-- • Ticket médio
-- • Prazo médio
-- • Taxa média de juros
-- • Valor total pago
-- • Valor em risco
-- • Contratos com atraso
-- • Contratos em default
-- • Taxa de atraso
-- • Taxa de default
--
-- Atualização..:
-- Full Refresh
--
-- Projeto......:
-- Credit Risk Lakehouse Brasil
-- =============================================================================
CREATE OR REPLACE TABLE dev_credit_risk.gold.mart_produto_credito
USING DELTA

AS

WITH carteira AS (

SELECT

    date_trunc('MONTH', dt_contratacao)          AS dt_competencia,

    id_produto,

    COUNT(*)                                     AS qtd_contratos,

    SUM(valor_contratado)                        AS valor_contratado,

    AVG(valor_contratado)                        AS ticket_medio,

    AVG(prazo_meses)                             AS prazo_medio,

    AVG(taxa_juros)                              AS taxa_media_juros

FROM dev_credit_risk.gold.fact_carteira_credito

GROUP BY

    date_trunc('MONTH',dt_contratacao),

    id_produto

),

cobranca AS (

SELECT

    dt_competencia,

    id_produto,

    SUM(valor_parcela)                           AS valor_esperado,

    SUM(COALESCE(valor_pago,0))                  AS valor_recebido,

    SUM(valor_em_aberto)                         AS saldo_em_aberto,

    ROUND(

        SUM(COALESCE(valor_pago,0))

        /

        NULLIF(SUM(valor_parcela),0)

    ,4)                                          AS indice_recebimento,

    ROUND(

        SUM(flag_pagamento_atrasado)

        /

        COUNT(*)

    ,4)                                          AS tx_atraso,

    ROUND(

        SUM(flag_pagamento_default_90)

        /

        COUNT(*)

    ,4)                                          AS tx_default,

    COUNT(

        DISTINCT CASE

            WHEN flag_pagamento_default_90=1

            THEN id_contrato

        END

    )                                            AS contratos_default

FROM dev_credit_risk.gold.fact_parcela_credito

GROUP BY

    dt_competencia,

    id_produto

)

SELECT

    c.dt_competencia,

    p.id_produto,

    p.modalidade_credito,

    c.qtd_contratos,

    c.valor_contratado,

    ROUND(

        c.valor_contratado

        /

        SUM(c.valor_contratado)

        OVER(PARTITION BY c.dt_competencia)

    ,4)                                          AS participacao_carteira,

    c.ticket_medio,

    c.prazo_medio,

    c.taxa_media_juros,

    b.valor_esperado,

    b.valor_recebido,

    b.indice_recebimento,

    b.saldo_em_aberto,

    b.contratos_default,

    b.tx_atraso,

    b.tx_default,

    current_timestamp()                          AS dt_carga

FROM carteira c

LEFT JOIN cobranca b

ON c.dt_competencia=b.dt_competencia

AND c.id_produto=b.id_produto

INNER JOIN dev_credit_risk.gold.dim_produto_credito p

ON c.id_produto=p.id_produto;

In [0]:
%sql
select * from dev_credit_risk.gold.mart_produto_credito